In [ ]:
!pip install numba

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import xmltodict
import rasterio
import os
from pathlib import Path
from dateutil.parser import isoparse
from scipy.interpolate import CubicHermiteSpline
from shapely.geometry import box
from eo_tools.util import show_cog
import rioxarray as rxr
import xarray as xr
from numba import jit, prange

In [ ]:
data_dir = "/data/S1"
product_dir = f"{data_dir}/S1A_IW_SLC__1SDV_20210624T172451_20210624T172518_038485_048A9C_A58F.SAFE"

In [ ]:
pth_tiff = Path(f"{product_dir}/measurement/").glob("*iw1*vv*.tiff")
pth_xml = Path(f"{product_dir}/annotation/").glob("*iw1*vv*.xml")
pth_tiff=list(pth_tiff)[0]
pth_xml=list(pth_xml)[0]

In [ ]:
with open(pth_xml) as f:
    meta = xmltodict.parse(f.read())

from rasterio.warp import calculate_default_transform, reproject

with rasterio.open(pth_tiff) as src:
    im = np.log(np.abs(src.read(1))+1)
    prof = src.profile.copy()
    # uncomment to reproject from GCPs
    gcps, src_crs = src.gcps
    dst_crs = "EPSG:4326"
    dst_trans, dst_width, dst_height = calculate_default_transform(
        src_crs=src.crs,
        dst_crs="EPSG:4326",
        width=src.width,
        height=src.height,
        gcps=src.gcps[0],
    )
    prof.update(
        dict(width=dst_width,
        height=dst_height,
        dtype="float32",
        crs="EPSG:4326",
        transform=dst_trans,
        nodata=0,
        driver="COG",
        compress="deflate",
        resampling="bilinear"
        )
    )
    im_burst = im.copy()
    im_burst[1500:] = 0
    with rasterio.open(f"/data/res/first_burst_geo.tif", "w", **prof) as dst:
        reproject(
            im_burst,
            rasterio.band(dst, 1),
            src_crs=src_crs,
            dst_crs=dst_crs,
            # src_transform=src.transform,
            gcps=gcps,
            dst_transform=dst_trans,
        )
        print(dst.profile)

In [ ]:
# show first bursts
plt.figure(figsize=(15,10))
vmin, vmax = np.percentile(im[im>0][::16], [10,90])
# plt.imshow(im[:3000][:,::4], vmin=vmin, vmax=vmax, cmap='grey')

In [ ]:
from eo_tools.util import show_cog
show_cog("/data/res/first_burst_geo.tif", rescale="2,9")

In [ ]:
# general info
image_info = meta["product"]["imageAnnotation"]["imageInformation"]
azimuth_time_interval = image_info["azimuthTimeInterval"]
slant_range_time = image_info["slantRangeTime"]
range_pixel_spacing = image_info["rangePixelSpacing"]
product_info = meta["product"]["generalAnnotation"]["productInformation"]
range_sampling_rate = product_info["rangeSamplingRate"]
radar_frequency = product_info["radarFrequency"]

In [ ]:
# look for burst info
burst_info = meta["product"]["swathTiming"]
lines_per_burst = burst_info["linesPerBurst"]
samples_per_burst = burst_info["samplesPerBurst"]
burst_count = burst_info["burstList"]["@count"]
first_burst = burst_info["burstList"]["burst"][0]
az_time = first_burst["azimuthTime"]
az_anx_time = first_burst["azimuthAnxTime"]

In [ ]:
# state vectors
orbit_list = meta["product"]["generalAnnotation"]["orbitList"]
orbit_count = orbit_list["@count"]
state_vectors = orbit_list["orbit"]

tt0 = isoparse(state_vectors[0]["time"])
tt = [(isoparse(it["time"]) - tt0).total_seconds() for it in state_vectors]
xx = [float(it["position"]["x"]) for it in state_vectors]
yy = [float(it["position"]["y"]) for it in state_vectors]
zz = [float(it["position"]["z"]) for it in state_vectors]

vxx = [float(it["velocity"]["x"]) for it in state_vectors]
vyy = [float(it["velocity"]["y"]) for it in state_vectors]
vzz = [float(it["velocity"]["z"]) for it in state_vectors]

interp_orb = CubicHermiteSpline(
    tt, np.array([xx, yy, zz]).T, np.array([vxx, vyy, vzz]).T
)
interp_orb_v = interp_orb.derivative(1)

In [ ]:
# dem download and conversion to ECEF
# found a bounding box based on gcps
import os
file_dem = "/data/dem_test_geocode.tif"

from eo_tools.dem import retrieve_dem

gcps_burst = [it for it in gcps if it.row <= 1500]

minmax  = lambda x: (x.min(), x.max())
xmin, xmax = minmax(np.array([p.x for p in gcps_burst]))
ymin, ymax = minmax(np.array([p.y for p in gcps_burst]))
shp = box(xmin, ymin, xmax, ymax)

if not os.path.exists(file_dem):
    retrieve_dem(shp, file_dem)


In [ ]:
# with rasterio.open(file_dem) as ds:
#     width, height = ds.width, ds.height
#     grid = np.meshgrid(np.arange(width), np.arange(height))
#     lat, lon = rasterio.transform.xy(ds.transform, grid[0].ravel(), grid[1].ravel())
#     alt = ds.read(1)

In [ ]:
# cropped image (faster)
with rasterio.open(file_dem) as ds:
    off_i, off_j = 500, 1500
    height, width = 100, 100
    dem_crs = ds.crs
    alt = ds.read(1)[off_i:off_i + height, off_j:off_j+width]
    grid = np.meshgrid(np.arange( off_j, off_j+width), np.arange(off_i, off_i + height))
    lat, lon = rasterio.transform.xy(ds.transform, grid[0].ravel(), grid[1].ravel())

In [ ]:
meta['product']['imageAnnotation']['imageInformation']['productFirstLineUtcTime'], az_time

In [ ]:
from rasterio.warp import transform
# flatten the DEM to get arrays of coordinates
WGS84_points = (np.array(lat), np.array(lon), alt.flat)
WGS84_crs = 'EPSG:4326+5773' 
ECEF_crs = 'EPSG:4978' # cartesian
dem_pts = transform(dem_crs, ECEF_crs, *WGS84_points)
# dem_pts = transform(WGS84_crs, ECEF_crs, *WGS84_points)
dem_pts = (np.array(dem_pts[0]), np.array(dem_pts[1]), np.array(dem_pts[2]))

In [ ]:
dem_pts[1], orb[1,1]

In [ ]:
t0_az = (isoparse(az_time) - tt0).total_seconds()
dt_az = float(azimuth_time_interval)
naz = int(lines_per_burst)
nrg = int(samples_per_burst)

In [ ]:
# compute orbit for each azimuth position
# pad the beginning and the end so that points not in azimuth validity zone are discarded
pad = 200
t_interp = np.linspace(t0_az - dt_az * pad, t0_az + dt_az * (naz + pad), naz + 2 * pad)
orb = interp_orb(t_interp)
orb_v = interp_orb_v(t_interp)

In [ ]:
# solve Range-Doppler equation
@jit(parallel=True)
def geocode_doppler(dem_x, dem_y, dem_z):
    az_min = np.zeros_like(dem_x)
    d2_min = np.zeros_like(dem_x)
    # look vectors
    # lv = np.zeros(dem_x.shape + (3,))

    # for a in prange(dem_x.shape[0]):
    for r in prange(dem_x.shape[0]):
        neg_doppler = False
        pos_doppler = False
        for t in np.arange(len(orb)):
            dx = dem_x[r] - orb[t, 0]
            dy = dem_y[r] - orb[t, 1]
            dz = dem_z[r] - orb[t, 2]
            d2 = dx ** 2 + dy ** 2 + dz ** 2
            doppler = - (orb_v[t, 0] * dx + orb_v[t, 1] * dy + orb_v[t, 2] * dz) / np.sqrt(d2)
            if doppler < 0:
                neg_doppler = True
            if doppler > 0:
                pos_doppler = True
            if neg_doppler and pos_doppler:
                break
        d2_min[r] = d2
        az_min[r] = t
        # lv[r, 0] = dx
        # lv[r, 1] = dy
        # lv[r, 2] = dz
    return az_min, d2_min #, lv

In [ ]:
az_geo, d2_geo = geocode_doppler(dem_pts[0], dem_pts[1], dem_pts[2])

In [ ]:
np.sqrt(d2_geo) / 1000

In [ ]:
r0 / 1000

In [ ]:
# convert range - azimuth to pixel indices
c0 = 299792458.0
r0 = float(slant_range_time) * c0 / 2
# dr = float(range_pixel_spacing)
dr = c0 / (2*float(range_sampling_rate))

rg_geo = ((np.sqrt(d2_geo)-r0)/dr)

cnd1 = (rg_geo >= 0) & (rg_geo < nrg)
cnd2 = (az_geo >= pad) & (az_geo < naz + pad)
valid = cnd1 & cnd2

rg = rg_geo.copy()
az = az_geo.copy() - pad

rg[~valid] = np.nan
# az[~valid] = np.nan

In [ ]:
dem_pts[2]

In [ ]:
dem_pts[1][0]

In [ ]:
plt.imshow(alt)

In [ ]:
dem_x = dem_pts[0]
dem_y = dem_pts[1]
dem_z = dem_pts[2]
dx = dem_x[50] - orb[:, 0]
dy = dem_y[50] - orb[:, 1]
dz = dem_z[50] - orb[:, 2]
d2 = dx*dx + dy*dy +dz*dz
doppler = - (orb_v[:, 0] * dx + orb_v[:, 1] * dy + orb_v[:, 2] * dz) / np.sqrt(d2)

In [ ]:
plt.plot(t_interp)